# Build Docs

Use the manifest-based profile loader to inspect available profiles and render graph diagrams for documentation or review.


In [2]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scoop").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the SCOOP repo root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_ROOT = ROOT / "scoop" / "configuration"
from IPython.display import HTML, display

from scoop.configuration.config_loader import list_profiles, load_profile, load_profile_manifest
from scoop.utils.document_generation import make_mermaid, mermaid_html


In [3]:
profiles = list_profiles(CONFIG_ROOT)
profiles


['bladder', 'nsclc', 'sclc']

In [4]:
manifest_summary = {}
for profile_id in profiles:
    manifest = load_profile_manifest(profile_id, base_dir=CONFIG_ROOT)
    manifest_summary[profile_id] = {
        'default_graph': manifest.default_graph,
        'graphs': sorted(manifest.graphs),
        'registry': str(manifest.regimen_registry),
    }

manifest_summary


{'bladder': {'default_graph': 'bladder',
  'graphs': ['bladder',
   'bladder_advanced_first_line',
   'bladder_advanced_post_platinum',
   'bladder_mibc_localized',
   'bladder_nmibc'],
  'registry': 'bladder_regimen_registry.yaml'},
 'nsclc': {'default_graph': 'nsclc',
  'graphs': ['nsclc',
   'nsclc_stg1',
   'nsclc_stg23_resectable',
   'nsclc_stg3_unresectable',
   'nsclc_stg4_driver',
   'nsclc_stg4_nodrivers'],
  'registry': 'nsclc_regimen_registry.yaml'},
 'sclc': {'default_graph': 'sclc',
  'graphs': ['sclc',
   'sclc_extensive',
   'sclc_limited_nonsurgical',
   'sclc_limited_surgical',
   'sclc_recurrent'],
  'registry': 'sclc_regimen_registry.yaml'}}

In [6]:
selected_profile = 'nsclc'
selected_graph = 'nsclc_stg4_driver'
ctx = load_profile(selected_profile, selected_graph, base_dir=CONFIG_ROOT)

display(HTML(ctx.graph.to_mermaid()))
